# 01 - Analisis Exploratorio de Datos (EDA)

**Dataset:** Diabetes 130-US Hospitals for years 1999-2008  
**Fuente:** UCI ML Repository - https://archive.ics.uci.edu/dataset/296  
**Problema:** Clasificacion binaria - Prediccion de reingreso hospitalario en < 30 dias  

**Referencia:**  
Strack et al. (2014). *Impact of HbA1c Measurement on Hospital Readmission Rates:  
Analysis of 70,000 Clinical Database Patient Records*. BioMed Research International.

In [3]:
import sys
print(sys.executable)

c:\Users\User\Desktop\mlops-alto-costo\.venv\Scripts\python.exe


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.max_columns', 50)

ModuleNotFoundError: No module named 'pandas'

## 1. Carga de Datos

Ejecutar primero: `uv run python src/data/download.py`

In [ ]:
df = pd.read_csv('../data/raw/diabetic_data.csv', na_values='?')
print(f'Shape: {df.shape[0]:,} filas x {df.shape[1]} columnas')
df.head(3)

## 2. Informacion General del Dataset

In [ ]:
print('=== TIPOS DE DATOS ===')
print(df.dtypes.value_counts())
print(f'\n=== VALORES NULOS (columnas con > 0 nulos) ===')
nulls = df.isnull().sum()
nulls_pct = (nulls / len(df) * 100).round(1)
null_df = pd.DataFrame({'nulos': nulls, 'porcentaje': nulls_pct})
print(null_df[null_df['nulos'] > 0].sort_values('porcentaje', ascending=False))

## 3. Analisis del Target (readmitted)

In [ ]:
target_counts = df['readmitted'].value_counts()
target_pct = df['readmitted'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2)

axes[0].bar(target_counts.index, target_counts.values, color=['steelblue','coral','forestgreen'])
axes[0].set_title('Distribucion de Reingreso')
axes[0].set_ylabel('Numero de encuentros')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontsize=10)

axes[1].pie(target_pct.values, labels=target_pct.index,
            autopct='%1.1f%%', colors=['steelblue','coral','forestgreen'])
axes[1].set_title('Proporcion de Reingreso')

plt.suptitle('Variable Target: readmitted', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nDesbalance de clases (target binario):')
df['target_bin'] = (df['readmitted'] == '<30').astype(int)
print(f'  Reingreso < 30 dias: {df["target_bin"].sum():,} ({df["target_bin"].mean():.1%})')
print(f'  No reingreso o > 30: {(1-df["target_bin"]).sum():,} ({1-df["target_bin"].mean():.1%})')
print(f'  Ratio desbalance: 1:{int(1/df["target_bin"].mean())}')

## 4. Variables Demograficas y Clinicas Clave

In [ ]:
# Tasa de reingreso por grupo de edad
age_order = ['[0-10)','[10-20)','[20-30)','[30-40)','[40-50)',
             '[50-60)','[60-70)','[70-80)','[80-90)','[90-100)']
readmit_by_age = df.groupby('age')['target_bin'].mean().reindex(age_order) * 100

fig, axes = plt.subplots(1, 2)

readmit_by_age.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Tasa de Reingreso < 30 dias por Edad')
axes[0].set_ylabel('Tasa de reingreso (%)')
axes[0].set_xlabel('Grupo de edad')
axes[0].tick_params(axis='x', rotation=45)

# Distribucion de dias en hospital
axes[1].hist(df['time_in_hospital'], bins=14, edgecolor='black', color='coral')
axes[1].set_title('Distribucion de Dias en Hospital')
axes[1].set_xlabel('Dias de hospitalizacion')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

## 5. Impacto de HbA1c (hallazgo del paper original)

In [ ]:
a1c_readmit = df.groupby('A1Cresult')['target_bin'].agg(['mean','count'])
a1c_readmit.columns = ['tasa_reingreso', 'n_pacientes']
a1c_readmit['tasa_reingreso'] = a1c_readmit['tasa_reingreso'] * 100

print('Tasa de reingreso < 30 dias segun resultado de HbA1c:')
print(a1c_readmit.sort_values('tasa_reingreso', ascending=False))
print('\nHallazgo del paper: medir HbA1c y encontrar valores altos (>8)\nse asocia con menor tasa de reingreso porque genera cambios en el tratamiento.')

## 6. Correlacion de Variables Numericas con el Target

In [ ]:
num_cols = ['time_in_hospital','num_lab_procedures','num_procedures',
            'num_medications','number_outpatient','number_emergency',
            'number_inpatient','number_diagnoses']

correlations = df[num_cols + ['target_bin']].corr()['target_bin'].drop('target_bin').sort_values()

plt.figure(figsize=(10, 5))
colors = ['tomato' if v < 0 else 'steelblue' for v in correlations.values]
plt.barh(correlations.index, correlations.values, color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Correlacion con reingreso < 30 dias')
plt.title('Correlacion de Variables Numericas con el Target')
plt.tight_layout()
plt.show()

print('\nTop 3 predictores positivos:')
print(correlations.tail(3))
print('\nTop 3 predictores negativos:')
print(correlations.head(3))

## 7. Conclusiones del EDA

1. **Desbalance severo:** Solo el ~11% de los encuentros son reingresos en < 30 dias.  
   Se requiere `scale_pos_weight` en XGBoost y umbral de decision ajustado (0.3).

2. **Reingresos previos** (`number_inpatient`) son el predictor mas fuerte de nuevo reingreso.  
   Los pacientes con historial de hospitalizaciones tienen 2-3x mas riesgo.

3. **HbA1c** (A1Cresult): paradojicamente, los pacientes con HbA1c > 8 tienen  
   *menor* tasa de reingreso porque el resultado activa cambios en el tratamiento.

4. **Polifarmacia** (`num_medications > 10`): presente en el ~35% de los pacientes,  
   es un predictor importante y justifica la feature de ingenieria `polypharmacy`.

5. **Columnas a eliminar:** `weight` (97% nulos), `payer_code` (40% nulos, no clinico),  
   `medical_specialty` (49% nulos), `examide` y `citoglipton` (varianza cero).

6. **Baseline esperado:** Un modelo que siempre predice 0 tiene 89% de accuracy  
   pero 0% de recall. La metrica relevante es ROC-AUC, objetivo > 0.68.